# Exploration: why does ranking by last quarter's decline fail?The one exploratory notebook in this repo. Everything reproducible lives in the CLI(`flightrisk build | backtest | falsify | text`) — this notebook exists only to show thework behind the project's headline finding, which was found by poking at a metric that madeno sense.**The anomaly.** The naive-persistence baseline — rank banks by how much their deposits felllast quarter — has a within-quarter ROC-AUC of **0.406**. That is *worse than random*. Yet itstop 1% has a precision lift of **3.49x**. Both numbers are correct, which means therelationship cannot be monotone.

In [ ]:
import pandas as pddf = pd.read_parquet("../data/processed/dataset.parquet")base = df["label"].mean()print(f"rows {len(df):,}   base rate {base:.4f}")

## The relationship is a U, not a line

In [ ]:
bins = [-1, -0.10, -0.05, -0.02, 0, 0.02, 0.05, 0.10, 10]names = ["<-10%", "-10..-5%", "-5..-2%", "-2..0%", "0..2%", "2..5%", "5..10%", ">10%"]d = df.dropna(subset=["dep_growth_1q"]).copy()d["bucket"] = pd.cut(d["dep_growth_1q"], bins=bins, labels=names)tbl = d.groupby("bucket", observed=True).agg(    rows=("label", "size"), next_qtr_drawdown_rate=("label", "mean"))tbl["vs_base"] = (tbl["next_qtr_drawdown_rate"] / base).round(2)tbl["rate_pct"] = (100 * tbl["next_qtr_drawdown_rate"]).round(2)tbl[["rows", "rate_pct", "vs_base"]]

Both tails are elevated. Banks that took in **more than 10% of new deposits** last quarter run a16.9% next-quarter drawdown rate — 3.0x the base rate, and nearly the 18.6% of banks that *lost*more than 10%. The safest place to be is boring: 0–2% growth, at 0.58x base.Ranking by decline therefore inverts across the middle of the distribution, which is exactly howyou get AUC below 0.5 alongside strong top-k precision.**Mechanism: lumpy money is transient money.** A large inflow is usually one depositor parking abalance that will move again — a municipality, a title company, a corporate treasury after araise. It shows up as growth, then leaves.

## Is it just small banks?

In [ ]:
d["asset"] = pd.to_numeric(d["ASSET"], errors="coerce")d["size"] = pd.cut(d["asset"], [0, 1e6, 1e7, 1e12], labels=["<$1B", "$1B-$10B", ">$10B"])piv = (    d.pivot_table(index="bucket", columns="size", values="label", aggfunc="mean", observed=True)    .div(base)    .round(2))piv  # each cell is lift vs the panel base rate

The U holds in every size band, so it is not a small-bank artifact.## Why this matters beyond banksThe transferable version: **an account that just received a large inflow is a drawdown risk, nota safe one.** For a startup banking platform, a customer that just closed a funding round isexactly the account whose balance is about to move — the opposite of the intuitive read.## What it cost the linear model`flightrisk falsify` quantifies this. Handing the U to an L1 logistic via fold-local quantilebins closes about half the in-sample gap to gradient boosting; binning every signal closes 85% ofthe *out-of-time* gap. The nonlinearity was the alpha — see `reports/recover_logit.csv`.